# MetaDiv Functional Ecology Module - ITS V4

This notebook generates site-level functional structural metrics from `Final_Database_ITS_FungalTraits_full_annotated.csv`.

Change only `DATASET_DIR` before running.

In [3]:
# =============================================================================
# Functional_Structural_Metrics_V4.ipynb
# =============================================================================
# MetaDiv Builder - ITS Functional Ecology Module V4
#
# Purpose
# -------
# Generate site-level fungal functional structural metrics directly from:
#
#   output/ITS/Functional_Ecology/
#   Final_Database_ITS_FungalTraits_full_annotated.csv
#
# Main strategies
# ---------------
# 1. Functional_Redundancy_Index
#    Taxonomic_Richness / Lifestyle_Richness
#
# 2. Functional_Compression_Difference
#    Lifestyle_Richness_Normalized - Generalized_Guild_Richness_Normalized
#
# 3. Functional_Compartmentalization_Index
#    Dominant_Guild_Richness / Taxonomic_Richness
#
# 4. Functional_Reorganization_Score
#    Exploratory composite score based on scaled independent metrics.
#
# V4 changes
# ----------
# - Removes duplicated Taxonomic_Functional_Decoupling_Ratio.
# - Renames Functional_Redundancy_Ratio as Functional_Redundancy_Index.
# - Renames Structural_Signal columns as Functional_Shift_Index.
# - Avoids pandas DataFrame fragmentation warnings.
# - Keeps the output table clearer and easier to interpret.
# =============================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime


# =============================================================================
# 1. USER CONFIGURATION
# =============================================================================

DATASET_DIR = Path(
    r"C:/Users/berna/Desktop/MetaDiv_Builder_V1_7_7/output/ITS"
)

FUNCTIONAL_DIR = DATASET_DIR / "Functional_Ecology"

INPUT_FILE = FUNCTIONAL_DIR / "Final_Database_ITS_FungalTraits_full_annotated_species_only.csv"

OUTPUT_DIR = FUNCTIONAL_DIR / "Functional_Structural_Metrics_V4"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_TABLE = OUTPUT_DIR / "Functional_Ecology_by_Site_V4_species.csv"
OUTPUT_LOG = OUTPUT_DIR / "LOG_Functional_Structural_Metrics_V4_species.txt"


# =============================================================================
# 2. COLUMN SETTINGS
# =============================================================================

SPPN_COL = "SPPN"

PRIMARY_CANDIDATES = [
    "FungalTraits_primary_lifestyle",
    "primary_lifestyle"
]

SECONDARY_CANDIDATES = [
    "FungalTraits_Secondary_lifestyle",
    "Secondary_lifestyle",
    "secondary_lifestyle"
]

GENERALIZED_TROPHIC_GUILDS = [
    "PATOTROPH",
    "SIMBIOTROPH",
    "SAPROTROPH",
    "UNCLASSIFIED",
    "SUPERPATOTROPH",
    "SUPERSIMBIOTROPH",
    "SUPERSAPROTROPH"
]

TOTAL_GENERALIZED_GUILDS = len(GENERALIZED_TROPHIC_GUILDS)

# Optional absolute lifestyle universe.
# Leave None to use the maximum Lifestyle_Richness observed in the dataset.
KNOWN_TOTAL_LIFESTYLES = None


# =============================================================================
# 3. LIFESTYLE TO GENERALIZED GUILD DICTIONARY
# =============================================================================

LIFESTYLE_TO_GENERALIZED_GUILD = {
    "algal_ectosymbiont": "SIMBIOTROPH",
    "algal_parasite": "PATOTROPH",
    "algal_symbiont": "SIMBIOTROPH",
    "algivorous/protistivorous": "SAPROTROPH",
    "animal-associated": "SIMBIOTROPH",
    "animal_decomposer": "SAPROTROPH",
    "animal_endosymbiont": "SIMBIOTROPH",
    "animal_parasite": "PATOTROPH",
    "arbuscular_mycorrhizal": "SIMBIOTROPH",
    "arthropod-associated": "SIMBIOTROPH",
    "arthropod_parasite": "PATOTROPH",
    "bacterivorous": "SAPROTROPH",
    "bryophilous": "SIMBIOTROPH",
    "coral-associated": "SIMBIOTROPH",
    "dung_saprotroph": "SAPROTROPH",
    "ectomycorrhizal": "SIMBIOTROPH",
    "epiphyte": "SIMBIOTROPH",
    "ericoid_mycorrhizal": "SIMBIOTROPH",
    "fatty_acid_producer": "SIMBIOTROPH",
    "fish_parasite": "PATOTROPH",
    "foliar_endophyte": "SIMBIOTROPH",
    "fungal_decomposer": "SAPROTROPH",
    "insect-associated": "SIMBIOTROPH",
    "invertebrate-associated": "SIMBIOTROPH",
    "invertebrate_parasite": "PATOTROPH",
    "lichen_parasite": "PATOTROPH",
    "lichenized": "SIMBIOTROPH",
    "litter_saprotroph": "SAPROTROPH",
    "liverwort-associated": "SIMBIOTROPH",
    "moss_parasite": "PATOTROPH",
    "moss_symbiont": "SIMBIOTROPH",
    "mycoparasite": "PATOTROPH",
    "myxomycete_decomposer": "SAPROTROPH",
    "nectar/tap_saprotroph": "SAPROTROPH",
    "nematophagous": "PATOTROPH",
    "plant_pathogen": "PATOTROPH",
    "pollen_saprotroph": "SAPROTROPH",
    "protistan_parasite": "PATOTROPH",
    "resin_saprotroph": "SAPROTROPH",
    "rock-inhabiting": "SAPROTROPH",
    "root-associated": "SIMBIOTROPH",
    "root_endophyte": "SIMBIOTROPH",
    "root_endophyte_dark_septate": "SIMBIOTROPH",
    "soil_saprotroph": "SAPROTROPH",
    "sooty_mold": "SAPROTROPH",
    "termite_symbiont": "SIMBIOTROPH",
    "unspecified_saprotroph": "SAPROTROPH",
    "unsepcified_saprotroph": "SAPROTROPH",
    "unspecified": "UNCLASSIFIED",
    "unspecified_pathotroph": "PATOTROPH",
    "unspecified_symbiotroph": "SIMBIOTROPH",
    "vertebrate-associated": "SIMBIOTROPH",
    "wood_saprotroph": "SAPROTROPH"
}


# =============================================================================
# 4. HELPER FUNCTIONS
# =============================================================================

def clean_text(value):
    """Clean text values for robust matching."""
    if pd.isna(value):
        return ""

    value = str(value).strip().lower()

    if value in ["", "nan", "none", "na", "n/a", "null"]:
        return ""

    return value


def split_lifestyles(value):
    """
    Split lifestyle annotations if multiple lifestyles are stored in one cell.

    Supported separators:
    - comma
    - semicolon
    - pipe
    """
    value = clean_text(value)

    if value == "":
        return []

    for sep in [";", "|"]:
        value = value.replace(sep, ",")

    return [
        x.strip()
        for x in value.split(",")
        if x.strip() not in ["", "nan", "none", "na", "n/a", "null"]
    ]


def classify_lifestyle(value):
    """Convert one lifestyle into one generalized base guild."""
    value = clean_text(value)

    if value == "":
        return "UNCLASSIFIED"

    return LIFESTYLE_TO_GENERALIZED_GUILD.get(value, "UNCLASSIFIED")


def classify_lifestyle_cell(value):
    """
    Convert one lifestyle cell into one generalized base guild.

    If multiple lifestyles occur in one cell, a conservative priority is used:
    1. PATOTROPH
    2. SIMBIOTROPH
    3. SAPROTROPH
    4. UNCLASSIFIED
    """
    lifestyles = split_lifestyles(value)

    if not lifestyles:
        return "UNCLASSIFIED"

    categories = [classify_lifestyle(x) for x in lifestyles]
    categories = [x for x in categories if x != "UNCLASSIFIED"]

    if not categories:
        return "UNCLASSIFIED"

    if "PATOTROPH" in categories:
        return "PATOTROPH"

    if "SIMBIOTROPH" in categories:
        return "SIMBIOTROPH"

    if "SAPROTROPH" in categories:
        return "SAPROTROPH"

    return "UNCLASSIFIED"


def classify_trophic_guild(primary_general, secondary_general):
    """
    Combine general_primary and general_secondary into one final generalized
    trophic guild.
    """
    p = str(primary_general).strip().upper()
    s = str(secondary_general).strip().upper()

    if p == "":
        p = "UNCLASSIFIED"

    if s == "":
        s = "UNCLASSIFIED"

    valid_base = {
        "PATOTROPH",
        "SIMBIOTROPH",
        "SAPROTROPH",
        "UNCLASSIFIED"
    }

    if p not in valid_base:
        p = "UNCLASSIFIED"

    if s not in valid_base:
        s = "UNCLASSIFIED"

    if p == "UNCLASSIFIED" and s == "UNCLASSIFIED":
        return "UNCLASSIFIED"

    if p != "UNCLASSIFIED" and s == "UNCLASSIFIED":
        return p

    if p == "UNCLASSIFIED" and s != "UNCLASSIFIED":
        return s

    if p == s:
        if p == "PATOTROPH":
            return "SUPERPATOTROPH"
        if p == "SIMBIOTROPH":
            return "SUPERSIMBIOTROPH"
        if p == "SAPROTROPH":
            return "SUPERSAPROTROPH"

    pair = {p, s}

    if "SIMBIOTROPH" in pair:
        return "SIMBIOTROPH"

    if "SAPROTROPH" in pair:
        return "SAPROTROPH"

    if "PATOTROPH" in pair:
        return "PATOTROPH"

    return "UNCLASSIFIED"


def minmax_normalize(series):
    """Scale a numeric series from 0 to 1."""
    series = pd.to_numeric(series, errors="coerce")

    min_value = series.min()
    max_value = series.max()

    if pd.isna(min_value) or pd.isna(max_value) or max_value == min_value:
        return pd.Series(np.nan, index=series.index)

    return (series - min_value) / (max_value - min_value)


def classify_quantile(series):
    """Classify values as LOW, MEDIUM, or HIGH using 33% and 66% quantiles."""
    series = pd.to_numeric(series, errors="coerce")

    q33 = series.quantile(0.33)
    q66 = series.quantile(0.66)

    def label(x):
        if pd.isna(x):
            return "UNDEFINED"
        if x <= q33:
            return "LOW"
        if x <= q66:
            return "MEDIUM"
        return "HIGH"

    return series.apply(label)


def shannon_evenness_simpson(counts):
    """Calculate Shannon diversity, Pielou evenness, and Simpson dominance."""
    counts = np.array(counts, dtype=float)

    total = counts.sum()

    if total <= 0:
        return np.nan, np.nan, np.nan

    proportions = counts / total
    proportions_nonzero = proportions[proportions > 0]

    shannon = -np.sum(proportions_nonzero * np.log(proportions_nonzero))

    if len(proportions_nonzero) > 1:
        evenness = shannon / np.log(len(proportions_nonzero))
    else:
        evenness = 0

    simpson_dominance = np.sum(proportions ** 2)

    return shannon, evenness, simpson_dominance


def find_first_existing_column(candidates, columns, label):
    """Find the first available column among candidate names."""
    for candidate in candidates:
        if candidate in columns:
            return candidate

    raise ValueError(
        f"No valid column found for {label}.\n"
        f"Expected one of:\n- " + "\n- ".join(candidates)
    )


# =============================================================================
# 5. LOAD INPUT
# =============================================================================

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Input file not found:\n{INPUT_FILE}")

print("Reading input file:")
print(INPUT_FILE)

df = pd.read_csv(INPUT_FILE, low_memory=False)

if SPPN_COL not in df.columns:
    raise ValueError(f"Missing required column: {SPPN_COL}")

PRIMARY_COL = find_first_existing_column(
    PRIMARY_CANDIDATES,
    df.columns,
    "primary lifestyle"
)

SECONDARY_COL = find_first_existing_column(
    SECONDARY_CANDIDATES,
    df.columns,
    "secondary lifestyle"
)

print(f"Using primary lifestyle column: {PRIMARY_COL}")
print(f"Using secondary lifestyle column: {SECONDARY_COL}")


# =============================================================================
# 6. DETECT SAMPLE COLUMNS
# =============================================================================

metadata_keywords = [
    "taxonomy",
    "domain",
    "kingdom",
    "phylum",
    "class",
    "order",
    "family",
    "genus",
    "species",
    "sequence",
    "sequence_lenght",
    "sequence_length",
    "sintax",
    "lifestyle",
    "guild",
    "comment",
    "fungaltraits",
    "template",
    "photobiont",
    "host",
    "growth",
    "fruitbody",
    "hymenium",
    "lineage",
    "substrate",
    "habitat",
    "capacity",
    "type",
    "confidence",
    "bootstrap",
    "rank",
    "sp_value",
    "sppn",
    "otu",
    "asv",
    "total_abundance"
]

sample_cols = []

for col in df.columns:
    col_lower = col.lower()

    if any(keyword in col_lower for keyword in metadata_keywords):
        continue

    numeric_col = pd.to_numeric(df[col], errors="coerce")

    if numeric_col.notna().sum() > 0:
        sample_cols.append(col)

if not sample_cols:
    raise ValueError("No sample abundance columns were detected.")

# Convert sample columns at once to avoid DataFrame fragmentation.
sample_numeric_df = (
    df[sample_cols]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
)

df_non_sample = df.drop(columns=sample_cols).copy()

df = pd.concat(
    [df_non_sample, sample_numeric_df],
    axis=1
).copy()

print(f"Detected sample columns: {len(sample_cols)}")


# =============================================================================
# 7. ADD FUNCTIONAL GUILD CLASSIFICATION WITHOUT FRAGMENTATION
# =============================================================================

functional_annotation = pd.DataFrame(index=df.index)

functional_annotation["general_primary"] = (
    df[PRIMARY_COL].apply(classify_lifestyle_cell)
)

functional_annotation["general_secondary"] = (
    df[SECONDARY_COL].apply(classify_lifestyle_cell)
)

functional_annotation["trophic_guild"] = [
    classify_trophic_guild(primary, secondary)
    for primary, secondary in zip(
        functional_annotation["general_primary"],
        functional_annotation["general_secondary"]
    )
]

df = pd.concat(
    [df.copy(), functional_annotation],
    axis=1
).copy()


# =============================================================================
# 8. SITE-LEVEL FUNCTIONAL STRUCTURAL METRICS
# =============================================================================

site_rows = []

for sample in sample_cols:

    present = df[df[sample] > 0].copy()

    taxonomic_richness = present[SPPN_COL].nunique()

    all_lifestyles = []

    for value in present[PRIMARY_COL]:
        all_lifestyles.extend(split_lifestyles(value))

    for value in present[SECONDARY_COL]:
        all_lifestyles.extend(split_lifestyles(value))

    all_lifestyles = [
        x for x in all_lifestyles
        if x not in ["", "unspecified", "nan", "none", "na", "n/a", "null"]
    ]

    lifestyle_richness = len(set(all_lifestyles))

    guild_richness = {}
    guild_abundance = {}
    guild_lifestyle_sets = {}

    for guild in GENERALIZED_TROPHIC_GUILDS:

        guild_data = present[present["trophic_guild"] == guild].copy()

        guild_richness[guild] = guild_data[SPPN_COL].nunique()
        guild_abundance[guild] = guild_data[sample].sum()

        lifestyles = []

        for value in guild_data[PRIMARY_COL]:
            lifestyles.extend(split_lifestyles(value))

        for value in guild_data[SECONDARY_COL]:
            lifestyles.extend(split_lifestyles(value))

        lifestyles = [
            x for x in lifestyles
            if x not in ["", "unspecified", "nan", "none", "na", "n/a", "null"]
        ]

        guild_lifestyle_sets[guild] = set(lifestyles)

    generalized_guild_richness = sum(
        1
        for guild in GENERALIZED_TROPHIC_GUILDS
        if guild_richness[guild] > 0
    )

    dominant_guild_by_richness = max(guild_richness, key=guild_richness.get)

    dominant_guild_richness = guild_richness[
        dominant_guild_by_richness
    ]

    functional_compartmentalization_index = (
        dominant_guild_richness / taxonomic_richness
        if taxonomic_richness > 0 else np.nan
    )

    guild_richness_counts = [
        guild_richness[guild]
        for guild in GENERALIZED_TROPHIC_GUILDS
    ]

    guild_shannon, guild_evenness, guild_simpson = shannon_evenness_simpson(
        guild_richness_counts
    )

    total_abundance = sum(guild_abundance.values())

    pathotroph_richness_compartment = (
        (
            guild_richness["PATOTROPH"]
            + guild_richness["SUPERPATOTROPH"]
        )
        / taxonomic_richness
        if taxonomic_richness > 0 else np.nan
    )

    saprotroph_richness_compartment = (
        (
            guild_richness["SAPROTROPH"]
            + guild_richness["SUPERSAPROTROPH"]
        )
        / taxonomic_richness
        if taxonomic_richness > 0 else np.nan
    )

    simbiotroph_richness_compartment = (
        (
            guild_richness["SIMBIOTROPH"]
            + guild_richness["SUPERSIMBIOTROPH"]
        )
        / taxonomic_richness
        if taxonomic_richness > 0 else np.nan
    )

    unclassified_richness_compartment = (
        guild_richness["UNCLASSIFIED"] / taxonomic_richness
        if taxonomic_richness > 0 else np.nan
    )

    pathotroph_relative_abundance = (
        (
            guild_abundance["PATOTROPH"]
            + guild_abundance["SUPERPATOTROPH"]
        )
        / total_abundance
        if total_abundance > 0 else np.nan
    )

    saprotroph_relative_abundance = (
        (
            guild_abundance["SAPROTROPH"]
            + guild_abundance["SUPERSAPROTROPH"]
        )
        / total_abundance
        if total_abundance > 0 else np.nan
    )

    simbiotroph_relative_abundance = (
        (
            guild_abundance["SIMBIOTROPH"]
            + guild_abundance["SUPERSIMBIOTROPH"]
        )
        / total_abundance
        if total_abundance > 0 else np.nan
    )

    unclassified_relative_abundance = (
        guild_abundance["UNCLASSIFIED"] / total_abundance
        if total_abundance > 0 else np.nan
    )

    pathotroph_functional_shift_index = (
        pathotroph_richness_compartment
        * pathotroph_relative_abundance
    )

    saprotroph_functional_shift_index = (
        saprotroph_richness_compartment
        * saprotroph_relative_abundance
    )

    simbiotroph_functional_shift_index = (
        simbiotroph_richness_compartment
        * simbiotroph_relative_abundance
    )

    lifestyle_counts_by_guild = {
        guild: len(guild_lifestyle_sets[guild])
        for guild in GENERALIZED_TROPHIC_GUILDS
    }

    lifestyle_counts_values = [
        lifestyle_counts_by_guild[guild]
        for guild in GENERALIZED_TROPHIC_GUILDS
    ]

    lifestyle_guild_shannon, lifestyle_guild_evenness, lifestyle_guild_simpson = (
        shannon_evenness_simpson(lifestyle_counts_values)
    )

    dominant_lifestyle_guild = max(
        lifestyle_counts_by_guild,
        key=lifestyle_counts_by_guild.get
    )

    dominant_lifestyle_guild_count = lifestyle_counts_by_guild[
        dominant_lifestyle_guild
    ]

    total_lifestyle_guild_assignments = sum(
        lifestyle_counts_by_guild.values()
    )

    lifestyle_compartmentalization_index = (
        dominant_lifestyle_guild_count / total_lifestyle_guild_assignments
        if total_lifestyle_guild_assignments > 0 else np.nan
    )

    row = {
        "SampleID": sample,

        # Core richness
        "Taxonomic_Richness": taxonomic_richness,
        "Lifestyle_Richness": lifestyle_richness,
        "Generalized_Guild_Richness": generalized_guild_richness,

        # Strategy 1: division
        "Functional_Redundancy_Index": (
            taxonomic_richness / lifestyle_richness
            if lifestyle_richness > 0 else np.nan
        ),

        # Strategy 2: normalized difference, calculated later
        "Functional_Compression_Difference": np.nan,

        # Strategy 3: compartmentalization
        "Functional_Compartmentalization_Index": functional_compartmentalization_index,
        "Dominant_Guild_By_Richness": dominant_guild_by_richness,
        "Dominant_Guild_Richness": dominant_guild_richness,

        # Alternative compartmentalization focused on lifestyles
        "Lifestyle_Compartmentalization_Index": lifestyle_compartmentalization_index,
        "Dominant_Lifestyle_Guild": dominant_lifestyle_guild,
        "Dominant_Lifestyle_Guild_Count": dominant_lifestyle_guild_count,

        # Guild distribution metrics
        "Guild_Richness_Shannon": guild_shannon,
        "Guild_Richness_Evenness": guild_evenness,
        "Guild_Richness_Simpson_Dominance": guild_simpson,

        # Lifestyle distribution across guilds
        "Lifestyle_Guild_Shannon": lifestyle_guild_shannon,
        "Lifestyle_Guild_Evenness": lifestyle_guild_evenness,
        "Lifestyle_Guild_Simpson_Dominance": lifestyle_guild_simpson,

        # Directional richness compartments
        "Pathotroph_Richness_Compartment": pathotroph_richness_compartment,
        "Saprotroph_Richness_Compartment": saprotroph_richness_compartment,
        "Simbiotroph_Richness_Compartment": simbiotroph_richness_compartment,
        "Unclassified_Richness_Compartment": unclassified_richness_compartment,

        # Directional abundance compartments
        "Pathotroph_Relative_Abundance": pathotroph_relative_abundance,
        "Saprotroph_Relative_Abundance": saprotroph_relative_abundance,
        "Simbiotroph_Relative_Abundance": simbiotroph_relative_abundance,
        "Unclassified_Relative_Abundance": unclassified_relative_abundance,

        # Directional shift indices
        "Pathotroph_Functional_Shift_Index": pathotroph_functional_shift_index,
        "Saprotroph_Functional_Shift_Index": saprotroph_functional_shift_index,
        "Simbiotroph_Functional_Shift_Index": simbiotroph_functional_shift_index
    }

    for guild in GENERALIZED_TROPHIC_GUILDS:
        row[f"{guild}_Richness"] = guild_richness[guild]
        row[f"{guild}_Abundance"] = guild_abundance[guild]
        row[f"{guild}_Lifestyle_Count"] = lifestyle_counts_by_guild[guild]

    site_rows.append(row)

site_df = pd.DataFrame(site_rows)


# =============================================================================
# 9. STRATEGY 2: FUNCTIONAL COMPRESSION DIFFERENCE
# =============================================================================

if KNOWN_TOTAL_LIFESTYLES is None:
    lifestyle_denominator = site_df["Lifestyle_Richness"].max()
    lifestyle_denominator_source = "Observed maximum Lifestyle_Richness"
else:
    lifestyle_denominator = KNOWN_TOTAL_LIFESTYLES
    lifestyle_denominator_source = "Known total lifestyle universe"

site_df["Lifestyle_Richness_Normalized"] = (
    site_df["Lifestyle_Richness"] / lifestyle_denominator
)

site_df["Generalized_Guild_Richness_Normalized"] = (
    site_df["Generalized_Guild_Richness"] / TOTAL_GENERALIZED_GUILDS
)

site_df["Functional_Compression_Difference"] = (
    site_df["Lifestyle_Richness_Normalized"]
    - site_df["Generalized_Guild_Richness_Normalized"]
)


# =============================================================================
# 10. CLASSES AND SCALED SCORES
# =============================================================================

metric_cols = [
    "Functional_Redundancy_Index",
    "Functional_Compression_Difference",
    "Functional_Compartmentalization_Index",
    "Lifestyle_Compartmentalization_Index",
    "Guild_Richness_Simpson_Dominance",
    "Lifestyle_Guild_Simpson_Dominance",
    "Pathotroph_Richness_Compartment",
    "Saprotroph_Richness_Compartment",
    "Simbiotroph_Richness_Compartment",
    "Pathotroph_Functional_Shift_Index",
    "Saprotroph_Functional_Shift_Index",
    "Simbiotroph_Functional_Shift_Index"
]

scaled_and_classes = pd.DataFrame(index=site_df.index)

for col in metric_cols:
    scaled_and_classes[f"{col}_Scaled_0_1"] = minmax_normalize(site_df[col])
    scaled_and_classes[f"{col}_Class"] = classify_quantile(site_df[col])

site_df = pd.concat(
    [site_df.copy(), scaled_and_classes],
    axis=1
).copy()


# =============================================================================
# 11. STRATEGY 4: FUNCTIONAL REORGANIZATION SCORE
# =============================================================================

site_df["Functional_Reorganization_Score"] = site_df[
    [
        "Functional_Redundancy_Index_Scaled_0_1",
        "Functional_Compression_Difference_Scaled_0_1",
        "Functional_Compartmentalization_Index_Scaled_0_1"
    ]
].mean(axis=1)

site_df["Functional_Reorganization_Class"] = classify_quantile(
    site_df["Functional_Reorganization_Score"]
)


# =============================================================================
# 12. CLEAN FINAL COLUMN ORDER
# =============================================================================

core_columns = [
    "SampleID",

    "Taxonomic_Richness",
    "Lifestyle_Richness",
    "Generalized_Guild_Richness",

    # Strategy 1
    "Functional_Redundancy_Index",
    "Functional_Redundancy_Index_Class",

    # Strategy 2
    "Lifestyle_Richness_Normalized",
    "Generalized_Guild_Richness_Normalized",
    "Functional_Compression_Difference",
    "Functional_Compression_Difference_Class",

    # Strategy 3
    "Functional_Compartmentalization_Index",
    "Functional_Compartmentalization_Index_Class",
    "Dominant_Guild_By_Richness",
    "Dominant_Guild_Richness",

    # Alternative lifestyle-focused compartmentalization
    "Lifestyle_Compartmentalization_Index",
    "Lifestyle_Compartmentalization_Index_Class",
    "Dominant_Lifestyle_Guild",
    "Dominant_Lifestyle_Guild_Count",

    # Distribution descriptors
    "Guild_Richness_Shannon",
    "Guild_Richness_Evenness",
    "Guild_Richness_Simpson_Dominance",
    "Guild_Richness_Simpson_Dominance_Class",

    "Lifestyle_Guild_Shannon",
    "Lifestyle_Guild_Evenness",
    "Lifestyle_Guild_Simpson_Dominance",
    "Lifestyle_Guild_Simpson_Dominance_Class",

    # Directional compartments
    "Pathotroph_Richness_Compartment",
    "Pathotroph_Richness_Compartment_Class",
    "Saprotroph_Richness_Compartment",
    "Saprotroph_Richness_Compartment_Class",
    "Simbiotroph_Richness_Compartment",
    "Simbiotroph_Richness_Compartment_Class",
    "Unclassified_Richness_Compartment",

    "Pathotroph_Relative_Abundance",
    "Saprotroph_Relative_Abundance",
    "Simbiotroph_Relative_Abundance",
    "Unclassified_Relative_Abundance",

    # Directional functional shifts
    "Pathotroph_Functional_Shift_Index",
    "Pathotroph_Functional_Shift_Index_Class",
    "Saprotroph_Functional_Shift_Index",
    "Saprotroph_Functional_Shift_Index_Class",
    "Simbiotroph_Functional_Shift_Index",
    "Simbiotroph_Functional_Shift_Index_Class",

    # Strategy 4
    "Functional_Reorganization_Score",
    "Functional_Reorganization_Class"
]

guild_columns = []

for guild in GENERALIZED_TROPHIC_GUILDS:
    guild_columns.extend([
        f"{guild}_Richness",
        f"{guild}_Abundance",
        f"{guild}_Lifestyle_Count"
    ])

final_columns = core_columns + guild_columns

site_df = site_df[final_columns].copy()

site_df = site_df.sort_values(
    by="Functional_Reorganization_Score",
    ascending=False
)


# =============================================================================
# 13. EXPORT TABLE
# =============================================================================

site_df.to_csv(
    OUTPUT_TABLE,
    index=False,
    encoding="utf-8-sig"
)


# =============================================================================
# 14. LOG FILE
# =============================================================================

log_text = f"""
MetaDiv Builder - ITS Functional Ecology Module V4
=================================================

Run date:
{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

Input file:
{INPUT_FILE}

Output table:
{OUTPUT_TABLE}

Purpose
-------
This module generates clean site-level fungal functional structural metrics
directly from the full FungalTraits annotated table exported by MetaDiv Builder.

The goal is to compare four complementary strategies for describing fungal
functional structure using ITS-derived functional annotations.

V4 changes
----------
1. Removed Taxonomic_Functional_Decoupling_Ratio because it duplicated
   Functional_Redundancy_Index.

2. Renamed Functional_Redundancy_Ratio as Functional_Redundancy_Index.

3. Renamed Structural_Signal columns as Functional_Shift_Index.

4. Avoided pandas DataFrame fragmentation warnings by adding derived columns
   using pd.concat().

5. Reorganized outputs around four clear strategies.

Four strategies
---------------

Strategy 1 — Functional_Redundancy_Index
Formula:
Taxonomic_Richness / Lifestyle_Richness

Meaning:
Average number of taxa represented per functional lifestyle.

High values:
High taxonomic redundancy per lifestyle. This may indicate mature or conserved
communities where many taxa support similar ecological functions.

Low values:
Low taxonomic redundancy per lifestyle. This may indicate taxonomic depletion,
low functional backup, or naturally specialized ecosystems.

Strategy 2 — Functional_Compression_Difference
Formula:
Lifestyle_Richness_Normalized - Generalized_Guild_Richness_Normalized

Where:
Lifestyle_Richness_Normalized = Lifestyle_Richness / {lifestyle_denominator}
Generalized_Guild_Richness_Normalized = Generalized_Guild_Richness / 7

Lifestyle denominator source:
{lifestyle_denominator_source}

Meaning:
Measures whether lifestyle diversity is proportionally higher or lower than
generalized guild diversity.

High positive values:
Many lifestyles are compressed into fewer generalized guild categories.

Strategy 3 — Functional_Compartmentalization_Index
Formula:
Dominant_Guild_Richness / Taxonomic_Richness

Meaning:
Measures whether taxonomic richness is concentrated inside one dominant trophic
guild.

High values:
One generalized guild contains a large fraction of the taxa detected in the
site.

Alternative lifestyle-focused compartmentalization
--------------------------------------------------
Lifestyle_Compartmentalization_Index:
Dominant_Lifestyle_Guild_Count / Total_Lifestyle_Guild_Assignments

Meaning:
Measures whether lifestyle diversity is concentrated inside one dominant
trophic guild.

Functional shift indices
------------------------
Pathotroph_Functional_Shift_Index:
Pathotroph_Richness_Compartment * Pathotroph_Relative_Abundance

Saprotroph_Functional_Shift_Index:
Saprotroph_Richness_Compartment * Saprotroph_Relative_Abundance

Simbiotroph_Functional_Shift_Index:
Simbiotroph_Richness_Compartment * Simbiotroph_Relative_Abundance

Meaning:
These metrics estimate whether richness and abundance are jointly shifted
toward pathotrophic, saprotrophic, or symbiotrophic strategies.

Strategy 4 — Functional_Reorganization_Score
Formula:
Mean of scaled:
- Functional_Redundancy_Index
- Functional_Compression_Difference
- Functional_Compartmentalization_Index

Meaning:
Exploratory composite score summarizing three independent dimensions:
division, normalized difference, and compartmentalization.

Warning:
This is not a final conservation or disturbance index unless validated against
independent metadata.

Classes
-------
LOW, MEDIUM, and HIGH are assigned using 33% and 66% quantiles within the
current dataset. These classes are dataset-relative.
"""

OUTPUT_LOG.write_text(log_text, encoding="utf-8")

print("Functional Structural Metrics V4 completed successfully.")
print(f"Output table:\n{OUTPUT_TABLE}")
print(f"Log file:\n{OUTPUT_LOG}")

display(site_df.head(25))


Reading input file:
C:\Users\berna\Desktop\MetaDiv_Builder_V1_7_7\output\ITS\Functional_Ecology\Final_Database_ITS_FungalTraits_full_annotated_species_only.csv
Using primary lifestyle column: FungalTraits_primary_lifestyle
Using secondary lifestyle column: FungalTraits_Secondary_lifestyle
Detected sample columns: 326
Functional Structural Metrics V4 completed successfully.
Output table:
C:\Users\berna\Desktop\MetaDiv_Builder_V1_7_7\output\ITS\Functional_Ecology\Functional_Structural_Metrics_V4\Functional_Ecology_by_Site_V4_species.csv
Log file:
C:\Users\berna\Desktop\MetaDiv_Builder_V1_7_7\output\ITS\Functional_Ecology\Functional_Structural_Metrics_V4\LOG_Functional_Structural_Metrics_V4_species.txt


,SampleID,Taxonomic_Richness,Lifestyle_Richness,Generalized_Guild_Richness,Functional_Redundancy_Index,Functional_Redundancy_Index_Class,Lifestyle_Richness_Normalized,Generalized_Guild_Richness_Normalized,Functional_Compression_Difference,Functional_Compression_Difference_Class,...,UNCLASSIFIED_Lifestyle_Count,SUPERPATOTROPH_Richness,SUPERPATOTROPH_Abundance,SUPERPATOTROPH_Lifestyle_Count,SUPERSIMBIOTROPH_Richness,SUPERSIMBIOTROPH_Abundance,SUPERSIMBIOTROPH_Lifestyle_Count,SUPERSAPROTROPH_Richness,SUPERSAPROTROPH_Abundance,SUPERSAPROTROPH_Lifestyle_Count
19,AtlasMxC105,118,1,2,118.000000,HIGH,0.030303,0.285714,-0.255411,LOW,...,0,0,0,0,0,0,0,0,0,0
289,X20_L__2_ITS2,2829,33,7,85.727273,HIGH,1.000000,1.000000,0.000000,HIGH,...,0,4,20,5,2,11,4,132,5324,10
303,X48_L__2_ITS2,2792,32,7,87.250000,HIGH,0.969697,1.000000,-0.030303,HIGH,...,0,3,15,5,4,6,4,106,3446,10
266,MXA_S665,2029,27,6,75.148148,HIGH,0.818182,0.857143,-0.038961,HIGH,...,0,0,0,0,4,4,2,35,133,9
285,X13_L__2_ITS2,2597,32,7,81.156250,HIGH,0.969697,1.000000,-0.030303,HIGH,...,0,6,50,6,1,4,2,119,3493,9
26,AtlasMxC116,1800,31,6,58.064516,HIGH,0.939394,0.857143,0.082251,HIGH,...,0,0,0,0,28,129,4,68,1430,9
317,X61_L__2_ITS2,2785,30,7,92.833333,HIGH,0.909091,1.000000,-0.090909,HIGH,...,0,3,10,5,8,18,4,95,5882,10
288,X18_L__2_ITS2,2056,33,7,62.303030,HIGH,1.000000,1.000000,0.000000,HIGH,...,0,3,16,3,9,27,2,72,4647,9
301,X46_L__2_ITS2,2560,31,7,82.580645,HIGH,0.939394,1.000000,-0.060606,HIGH,...,0,5,28,2,12,63,3,90,2434,9
309,X54_L__2_ITS2,2619,29,7,90.310345,HIGH,0.878788,1.000000,-0.121212,HIGH,...,0,5,550,4,6,18,2,77,2613,10
